In [ ]:
!pip install langchain langchain-community langchain-experimental gradio neo4j pandas requests
!pip install -U langchain-ollama

In [1]:
from torch_geometric.data import HeteroData
from collections import defaultdict
import random
import numpy as np
from datetime import datetime
import ast  # To safely convert string representations of lists to actual lists
import re
from langchain.chains import GraphCypherQAChain
import torch.nn.functional as F
from torch_geometric.nn import (to_hetero, GraphConv, GATConv, GCNConv, SAGEConv, GATv2Conv, Linear, HeteroConv, HGTConv, RGCNConv, RGATConv, MessagePassing, global_add_pool)
from torch_geometric.loader import NeighborLoader
import torch_geometric.transforms as T
from torch_geometric.explain import GNNExplainer
import torch_geometric
import pyg_lib
import torch_sparse
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torch_geometric.utils import degree


import os
import requests
import pandas as pd
from neo4j import GraphDatabase
from langchain.graphs import Neo4jGraph
from langchain.docstore.document import Document
#from langchain_core.documents import Document
from langchain_text_splitters import TokenTextSplitter
from langchain_ollama import OllamaLLM
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Neo4jVector
from langchain_community.graphs import Neo4jGraph
from langchain.chains import RetrievalQA
import gradio as gr
from dotenv import load_dotenv
import torch



print("PyTorch Version:", torch.__version__)
print("PyG Loaded Successfully!")

PyTorch Version: 2.5.1+cu121
PyG Loaded Successfully!


In [2]:

# Load environment variables from .env file
load_dotenv()

# Set seed for reproducibility
def set_seed(seed_value=24):
    torch.manual_seed(seed_value)
    random.seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    np.random.seed(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(24)

In [ ]:
PTH = os.getenv("PTH")

x = os.getenv("user_profile")

# bartala
NEO4J_USERNAME = os.getenv("NEO4J_USER")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

NEO4J_URI = os.getenv("NEO4J_URI_"+x)
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD_"+x)

print(NEO4J_URI)

neo4j+s://35e8b0ff.databases.neo4j.io


In [8]:
graph = Neo4jGraph(
    url = NEO4J_URI,
    username = NEO4J_USERNAME,
    password = NEO4J_PASSWORD,
    database = NEO4J_DATABASE
)

In [ ]:

graph.query("""
CREATE CONSTRAINT unique_document IF NOT EXISTS
FOR (d:Document) REQUIRE d.id IS UNIQUE
""")


graph.query("""
CREATE VECTOR INDEX document_embedding_index IF NOT EXISTS
FOR (d:Document) ON (d.textEmbedding)
OPTIONS {
  indexConfig: {
    `vector.dimensions`: 1536,
    `vector.similarity_function`: 'cosine'
  }
}
""")

In [ ]:
# Load into DataFrame
df = pd.read_csv(os.path.join(PTH,'embedded_CBPTSD.csv'))

df = df[df['source']=='CBEx']

In [18]:
# Format the dataset into documents
documents = []
for index, row in df.iterrows():
    doc = Document(
      page_content= row['cb_delivery_narrative'],
      metadata={
           'id' : row['record_id'],
           'spcl5_total' : row['spcl5_total'],
           'pdi_total' : row['pdi_total'],
           'source' : row['source'],
           'n_words' : row['n_words'],
           'ptsd' : row['y']
      }
    )
    documents.append(doc)

In [ ]:
# Initialize LLM
llm = OllamaLLM(model="llama3")

# Create nodes and edges using LLMGraphTransformer
llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=[],
    allowed_relationships=[],
    )

graph_documents = llm_transformer.convert_to_graph_documents(documents)

In [ ]:
# Store graph documents in Neo4j
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)